# Data preprocessing

## Experimental Setup & Methodology

**Task:** Lipophilicity (logD) prediction on MolNet Lipophilicity dataset (~4,200 molecules)

**Dataset:** Bemis-Murcko scaffold split (training, validation, test) via DeepChem

**Representations & Models:**

| ID | Representation | Dimensionality | Model | CV Strategy |
|----|---|---|---|---|
| M2D | Mordred 2D descriptors | ~1,600 → ~50-100 (RFECV) | Random Forest (1500 trees) | 5-fold scaffold CV |
| M3D | Mordred 3D descriptors | ~1,800 → ~50-100 (RFECV) | Random Forest (1500 trees) | 5-fold scaffold CV |
| MACCS | MACCS keys | 167 bits | SGDRegressor (L2) | 5-fold scaffold CV |
| ECFP | Morgan/ECFP (r=2, 512 bits) | 512 bits | SGDRegressor (L2) | 5-fold scaffold CV |
| ERG | Gobbi Pharm2D → TruncatedSVD | 39K sparse → 256 dims | SGDRegressor (L2) + SVD | 5-fold scaffold CV |
| M2V | Mol2Vec (pretrained, 300-dim) | 300 | SVR (RBF, C=10) | 5-fold scaffold CV |
| CB-PT | ChemBERTa (pretrained) | 768 | SVR (RBF, C=10) | 5-fold scaffold CV |
| CB-FT | ChemBERTa (MLM fine-tuned) | 768 | SVR (RBF, C=10) | Official split only* |

*Fine-tuned on training split SMILES; full-data CV would leak training corpus into val folds.

**Evaluation Metrics:** MAE, RMSE, R² (on untransformed original scale)
- All predictions untransformed using DeepChem's NormalizationTransformer

**Fixes Implemented in This Notebook:**

1. ✅ **Fixed 3D Mordred bug:** Missing `continue` after failed molecule optimization exception caused data corruption.
2. ✅ **Fixed column leakage:** Mordred descriptor cleaning now uses train-only decisions; val/test apply same rules.
3. ✅ **Removed function shadowing:** Single `murcko_scaffold` definition; unified `scaffold_cv` for all representations.
4. ✅ **Proper variable scoping:** Each representation has prefixed variables (X_m2d_*, X_m3d_*, X_m2v_*, etc.) to prevent overwrites.
5. ✅ **Added R² metric:** Standard in MolNet benchmarks; included in all evaluations.
6. ✅ **Unified evaluation:** All models use `untransform_and_evaluate()` helper (consistent, reproducible).
7. ✅ **Addressed CV leakage:** Fine-tuned ChemBERTa evaluated on official split only (documented limitation).
8. ✅ **Reproducibility:** All random operations use RANDOM_STATE=42; seeds set at import.

Loading the MolNet Lipophilicity dataset which contains 4200 molecules.

### <font color='red'> You must undo this when reporting predictions: </font>

y_pred = model.predict(test_dataset)

y_pred = transformers[0].untransform(y_pred)

Also, this exists:

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)

model.evaluate(test_dataset, [metric], transformers)

In [53]:
# ============================================================================
# CONSOLIDATED IMPORTS & GLOBAL CONFIGURATION
# ============================================================================

# Core libraries
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# RDKit
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.DataStructs.cDataStructs import ConvertToNumpyArray
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D
RDLogger.DisableLog('rdApp.*')

# scikit-learn
from sklearn.model_selection import GroupKFold
from sklearn.feature_selection import VarianceThreshold, RFECV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

# DeepChem (for dataset loading)
import deepchem as dc

# Molecular representations
from mordred import Calculator, descriptors
import gensim
from mol2vec.features import mol2alt_sentence

# Deep learning
import torch
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
)
from datasets import Dataset

# ============================================================================
# GLOBAL CONFIGURATION
# ============================================================================

RANDOM_STATE = 42
N_JOBS = -1
N_SPLITS = 5

# Set seeds for reproducibility
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [54]:
# ============================================================================
# UNIFIED UTILITY FUNCTIONS (for evaluation and CV)
# ============================================================================

def untransform_and_evaluate(y_true, y_pred, transformers, prefix="", verbose=True):
    """
    Untransform predictions and labels, then compute MAE, RMSE, R².
    
    Args:
        y_true: ground truth labels
        y_pred: predictions
        transformers: list of deepchem transformers (from dc.molnet.load_lipo)
        prefix: optional prefix for print statements
        verbose: whether to print results
    
    Returns:
        dict with 'mae', 'rmse', 'r2'
    """
    y_true_u = np.array(y_true).reshape(-1, 1)
    y_pred_u = np.array(y_pred).reshape(-1, 1)
    
    for transformer in transformers:
        if getattr(transformer, "transform_y", False):
            y_true_u = transformer.untransform(y_true_u)
            y_pred_u = transformer.untransform(y_pred_u)
    
    y_true_u = y_true_u.ravel()
    y_pred_u = y_pred_u.ravel()
    
    mae = mean_absolute_error(y_true_u, y_pred_u)
    rmse = mean_squared_error(y_true_u, y_pred_u, squared=False)
    r2 = r2_score(y_true_u, y_pred_u)
    
    if verbose:
        print(f"{prefix}MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")
    
    return {"mae": mae, "rmse": rmse, "r2": r2}


def murcko_scaffold(smiles: str) -> str:
    """Return Bemis-Murcko scaffold for a single SMILES string."""
    if not isinstance(smiles, str):
        return "[invalid_smiles]"
    smiles = smiles.strip()
    if not smiles:
        return "[invalid_smiles]"
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles, includeChirality=False)
    except ValueError:
        return "[invalid_smiles]"


def scaffold_cv(X_all, y_all, smiles_all, make_model_fn, transformers, 
                n_splits=N_SPLITS, descriptor_selection_fn=None, verbose=True):
    """
    Scaffold-aware cross-validation with optional feature selection per fold.
    
    Args:
        X_all: feature matrix (numpy array or DataFrame)
        y_all: target array
        smiles_all: array of SMILES strings
        make_model_fn: callable that returns a new unfitted model
        transformers: list of deepchem transformers
        n_splits: number of scaffold CV folds
        descriptor_selection_fn: optional function(Xtr, ytr, groups_tr, Xva, Xte) 
                                  -> (Xtr_sel, Xva_sel, Xte_sel, info)
                                  If provided, called per fold to select features
        verbose: whether to print per-fold results
    
    Returns:
        dict with 'maes', 'rmses', 'r2s' (lists), and 'mean_mae', 'std_mae', 'mean_rmse', 'std_rmse', etc.
    """
    groups_all = np.array([murcko_scaffold(s) for s in smiles_all])
    cv = GroupKFold(n_splits=n_splits)
    
    maes, rmses, r2s = [], [], []
    
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_all, y_all, groups=groups_all), 1):
        # Extract fold data
        if isinstance(X_all, pd.DataFrame):
            Xtr, Xva = X_all.iloc[tr_idx], X_all.iloc[va_idx]
        else:
            Xtr, Xva = X_all[tr_idx], X_all[va_idx]
        
        ytr, yva = np.asarray(y_all)[tr_idx], np.asarray(y_all)[va_idx]
        groups_tr = groups_all[tr_idx]
        
        # Optional per-fold feature selection
        if descriptor_selection_fn is not None:
            Xtr_sel, Xva_sel, _, info = descriptor_selection_fn(Xtr, ytr, groups_tr, Xva, Xva)
        else:
            Xtr_sel, Xva_sel = Xtr, Xva
        
        # Train and predict
        model = make_model_fn()
        model.fit(Xtr_sel, ytr)
        pred = model.predict(Xva_sel)
        
        # Evaluate
        metrics = untransform_and_evaluate(yva, pred, transformers, prefix=f"Fold {fold}: ", verbose=verbose)
        maes.append(metrics["mae"])
        rmses.append(metrics["rmse"])
        r2s.append(metrics["r2"])
    
    # Summary statistics
    if verbose:
        print(f"\n{n_splits}-Fold Scaffold CV Summary:")
        print(f"MAE  mean±std:  {np.mean(maes):.4f} ± {np.std(maes):.4f}")
        print(f"RMSE mean±std:  {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
        print(f"R²   mean±std:  {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
    
    return {
        "maes": maes,
        "rmses": rmses,
        "r2s": r2s,
        "mean_mae": np.mean(maes),
        "std_mae": np.std(maes),
        "mean_rmse": np.mean(rmses),
        "std_rmse": np.std(rmses),
        "mean_r2": np.mean(r2s),
        "std_r2": np.std(r2s),
    }

In [55]:
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_lipo(
    featurizer='ecfp',  # use ECFP featurizer
    splitter='scaffold',
    reload=False  # disable loading from cached directory
)

In [56]:
(train, val, test) = datasets
# train.ids = smiles; train.y = labels

len(train), len(val), len(test)

(3360, 420, 420)

In [57]:
# canonizalization of SMILES
from rdkit import Chem
import numpy as np

def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

train_smiles = [canonicalize(smi) for smi in train.ids]
val_smiles = [canonicalize(smi) for smi in val.ids]
test_smiles = [canonicalize(smi) for smi in test.ids]

train_y = train.y.reshape(-1)
val_y   = val.y.reshape(-1)
test_y  = test.y.reshape(-1)

In [58]:
# checking whether there is any missing data

import pandas as pd

data_train = pd.DataFrame({'smiles': train_smiles, 'target': train_y.astype(float)})
data_val = pd.DataFrame({'smiles': val_smiles, 'target': val_y.astype(float)})
data_test = pd.DataFrame({'smiles': test_smiles, 'target': test_y.astype(float)})

data = pd.concat([data_train, data_val, data_test], ignore_index=True)

data.isna().sum()

smiles    0
target    0
dtype: int64

No need to manually scale, that is done while loading the dataset with DeepChem. 

It is important to untransform predictions for reporting.

DeepChem also already handles outliers: standardizes targets using meand & std, does not assume Gaussian tails, leaves the distribution intact

In [59]:
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np
from sklearn.model_selection import GroupKFold

def murcko_scaffolds(smiles_list):
    """Vectorized wrapper around the shared single-SMILES scaffold helper."""
    return np.array([murcko_scaffold(s) for s in smiles_list])

# Molecular representations

In [60]:
# !pip install rdkit-pypi mordred pandas numpy

## Molecular Descriptors

In [61]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from mordred import Calculator, descriptors


# ============================================================================
# FUNCTIONS: MORDRED DESCRIPTOR COMPUTATION & CLEANING
# ============================================================================

def fit_mordred_cleaner(raw_train_df):
    """
    Determine which columns to keep based on training data only.
    
    1. Converts to numeric (NaN for invalid)
    2. Replaces inf with NaN
    3. Drops columns with >80% missing values
    
    Args:
        raw_train_df: raw Mordred DF with 'smiles' and 'target' columns
    
    Returns:
        tuple: (clean_train_df, list of kept column names)
    """
    X = raw_train_df.drop(columns=["smiles", "target"]).copy()
    y = raw_train_df["target"].copy()
    smiles = raw_train_df["smiles"].copy()

    # convert to numeric; invalid parsing -> NaN
    X = X.apply(pd.to_numeric, errors='coerce')

    # replace inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # drop columns with too many missing values (threshold = 80% non-missing)
    threshold = 0.8 * len(X)
    X = X.dropna(axis=1, thresh=threshold)
    kept = X.columns.tolist()

    # drop rows with any missing values
    row_mask = X.notna().all(axis=1)
    X_clean = X.loc[row_mask].copy()
    y_clean = y.loc[row_mask].copy()
    smiles_clean = smiles.loc[row_mask].copy()

    clean_df = X_clean.copy()
    clean_df.insert(0, "smiles", smiles_clean)
    clean_df.insert(1, "target", y_clean)

    return clean_df, kept


def apply_mordred_cleaner(raw_df, kept_cols):
    """
    Apply the same cleaning rules (column & row mask) to any split.
    
    Args:
        raw_df: raw Mordred DF with 'smiles' and 'target' columns
        kept_cols: list of columns to keep (from fit_mordred_cleaner)
    
    Returns:
        clean_df with same columns as training, rows with any NaN dropped
    """
    X = raw_df.drop(columns=["smiles", "target"]).copy()
    y = raw_df["target"].copy()
    smiles = raw_df["smiles"].copy()

    # convert to numeric; invalid parsing -> NaN
    X = X.apply(pd.to_numeric, errors='coerce')

    # replace inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # select only the columns from training
    X = X[kept_cols].copy()

    # drop rows with any missing values
    row_mask = X.notna().all(axis=1)
    X_clean = X.loc[row_mask].copy()
    y_clean = y.loc[row_mask].copy()
    smiles_clean = smiles.loc[row_mask].copy()

    clean_df = X_clean.copy()
    clean_df.insert(0, "smiles", smiles_clean)
    clean_df.insert(1, "target", y_clean)

    return clean_df


# function to compute 2D Mordred descriptors
def compute_mordred_2d(df):
    mols =  []
    good_smiles =  []
    good_y = []
    bad_smiles = []

    for _, row in df.iterrows():
        s = row['smiles']
        m = Chem.MolFromSmiles(s)
        if m is None:
            bad_smiles.append(s)
        else:
            mols.append(m)
            good_smiles.append(s)
            good_y.append(row['target'])
    print("Valid:", len(good_smiles), "Invalid:", len(bad_smiles))

    calc = Calculator(descriptors, ignore_3D=True)
    mordred_df = calc.pandas(mols)
    mordred_df.insert(0, "smiles", good_smiles)
    mordred_df.insert(1, "target", good_y)

    print("Number of 2D descriptors computed:", mordred_df.shape[1] - 2)
    return mordred_df


# function to compute 3D Mordred descriptors
def compute_mordred_3d(df):
    mols = []
    good_smiles = []
    good_y = []
    bad_smiles = []

    params = AllChem.ETKDGv3()
    params.randomSeed = RANDOM_STATE  # for reproducibility

    for _, row in df.iterrows():
        s = row['smiles']
        m = Chem.MolFromSmiles(s)
        if m is None:
            bad_smiles.append(s)
            continue  # Skip invalid SMILES
        
        # Generate 3D coordinates
        m_3d = Chem.AddHs(m)
        coded = AllChem.EmbedMolecule(m_3d, params)
        if coded == 0:
            try:
                if AllChem.MMFFHasAllMoleculeParams(m_3d):
                    AllChem.MMFFOptimizeMolecule(m_3d)
                else:
                    AllChem.UFFOptimizeMolecule(m_3d)
                mols.append(m_3d)
                good_smiles.append(s)
                good_y.append(row['target'])
            except Exception as e:
                bad_smiles.append(s)
                print(f"Optimize failed for SMILES: {s}, Error: {e}")
                continue  # BUG FIX: Skip failed molecules entirely
        else:
            bad_smiles.append(s)
    
    print("Valid:", len(good_smiles), "Invalid:", len(bad_smiles))

    calc3d = Calculator(descriptors, ignore_3D=False)
    mordred_3d_df = calc3d.pandas(mols)
    mordred_3d_df.insert(0, "smiles", good_smiles)
    mordred_3d_df.insert(1, "target", good_y)

    print("Number of 3D descriptors computed:", mordred_3d_df.shape[1] - 2)
    return mordred_3d_df

For now, I will only use 2d descriptors since the computation of 3D descriptors takes a substantial amount of time. I will compute 3D descriptors later and validate them on the validation set to choose a better fit.

Generation of 2D molecular descriptors takes around 1.30 minutes. Generation od 3D molecular descriptors takes around 8 minutes.

In [62]:
# ============================================================================
# LOAD OR COMPUTE 2D DESCRIPTORS (with train-only cleaning)
# ============================================================================

# Uncomment and run the following lines only once to generate raw descriptors
# (then save to cache for subsequent runs):
raw_mordred_2d_train = compute_mordred_2d(data_train)
raw_mordred_2d_val = compute_mordred_2d(data_val)
raw_mordred_2d_test = compute_mordred_2d(data_test)

raw_mordred_2d_train.to_pickle("./cache/mordred_2d_lipo_train_raw.pkl")
raw_mordred_2d_val.to_pickle("./cache/mordred_2d_lipo_val_raw.pkl")
raw_mordred_2d_test.to_pickle("./cache/mordred_2d_lipo_test_raw.pkl")

# Load raw descriptors and apply train-only cleaning
# raw_mordred_2d_train = pd.read_pickle("./cache/mordred_2d_lipo_train.pkl")  # TODO: rename to _raw.pkl
# raw_mordred_2d_val = pd.read_pickle("./cache/mordred_2d_lipo_val.pkl")      # TODO: rename to _raw.pkl
# raw_mordred_2d_test = pd.read_pickle("./cache/mordred_2d_lipo_test.pkl")    # TODO: rename to _raw.pkl

# Fit cleaner on training data only; apply to all splits
mordred_df_2d_train, kept_2d_cols = fit_mordred_cleaner(raw_mordred_2d_train)
mordred_df_2d_val = apply_mordred_cleaner(raw_mordred_2d_val, kept_2d_cols)
mordred_df_2d_test = apply_mordred_cleaner(raw_mordred_2d_test, kept_2d_cols)

print(f"2D Mordred: {len(kept_2d_cols)} features after cleaning")
print(f"  Train: {len(mordred_df_2d_train)} molecules, Val: {len(mordred_df_2d_val)}, Test: {len(mordred_df_2d_test)}")

Valid: 3360 Invalid: 0


 79%|███████▉  | 2668/3360 [00:45<00:13, 52.02it/s]

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 3360/3360 [01:00<00:00, 55.24it/s]


Number of 2D descriptors computed: 1613
Valid: 420 Invalid: 0


...

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 33%|███▎      | 137/420 [00:04<00:04, 65.08it/s]

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 420/420 [00:09<00:00, 46.02it/s]


c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
Number of 2D descriptors computed: 1613
2D Mordred: 1429 features after cleaning
  Train: 2649 molecules, Val: 329, Test: 343


In [63]:
# ============================================================================
# LOAD OR COMPUTE 3D DESCRIPTORS (with train-only cleaning)
# ============================================================================

# Uncomment and run the following lines only once to generate raw descriptors
# (takes ~8 minutes):
raw_mordred_3d_train = compute_mordred_3d(data_train)
raw_mordred_3d_val = compute_mordred_3d(data_val)
raw_mordred_3d_test = compute_mordred_3d(data_test)

raw_mordred_3d_train.to_pickle("./cache/mordred_3d_lipo_train_raw.pkl")
raw_mordred_3d_val.to_pickle("./cache/mordred_3d_lipo_val_raw.pkl")
raw_mordred_3d_test.to_pickle("./cache/mordred_3d_lipo_test_raw.pkl")

# Load raw descriptors and apply train-only cleaning
# raw_mordred_3d_train = pd.read_pickle("./cache/mordred_3d_lipo_train.pkl")  # TODO: rename to _raw.pkl
# raw_mordred_3d_val = pd.read_pickle("./cache/mordred_3d_lipo_val.pkl")      # TODO: rename to _raw.pkl
# raw_mordred_3d_test = pd.read_pickle("./cache/mordred_3d_lipo_test.pkl")    # TODO: rename to _raw.pkl

# Fit cleaner on training data only; apply to all splits
mordred_df_3d_train, kept_3d_cols = fit_mordred_cleaner(raw_mordred_3d_train)
mordred_df_3d_val = apply_mordred_cleaner(raw_mordred_3d_val, kept_3d_cols)
mordred_df_3d_test = apply_mordred_cleaner(raw_mordred_3d_test, kept_3d_cols)

print(f"3D Mordred: {len(kept_3d_cols)} features after cleaning")
print(f"  Train: {len(mordred_df_3d_train)} molecules, Val: {len(mordred_df_3d_val)}, Test: {len(mordred_df_3d_test)}")

Valid: 3359 Invalid: 1


100%|██████████| 3359/3359 [01:26<00:00, 38.66it/s]


Number of 3D descriptors computed: 1826
Valid: 419 Invalid: 1


 74%|███████▍  | 311/419 [00:11<00:09, 11.22it/s]

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 61%|██████    | 256/420 [00:14<00:06, 24.93it/s]

c:\Users\afs\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 420/420 [00:19<00:00, 21.01it/s]


Number of 3D descriptors computed: 1826
3D Mordred: 1482 features after cleaning
  Train: 2648 molecules, Val: 328, Test: 343


### Scaffolding functions

In [64]:
# ============================================================================
# FEATURE SELECTION UTILITIES FOR MORDRED DESCRIPTORS
# ============================================================================
# (Imports consolidated at top; murcko_scaffold defined in utilities cell)

def drop_autocorr_cols(X: pd.DataFrame, keywords=("ATSC", "AATS")) -> pd.DataFrame:
    """Drop autocorrelation descriptor columns."""
    mask = X.columns.str.contains("|".join(keywords))
    return X.loc[:, ~mask]


def apply_variance_threshold_train(Xtr: pd.DataFrame, Xva: pd.DataFrame, Xte: pd.DataFrame,
                                   threshold: float = 1e-8):
    """Fit variance threshold on training data; apply to all splits."""
    vt = VarianceThreshold(threshold=threshold)
    vt.fit(Xtr)
    kept = Xtr.columns[vt.get_support()].tolist()
    return Xtr[kept], Xva[kept], Xte[kept], kept


def prune_correlated_features_by_importance(X: pd.DataFrame, y,
                                           threshold: float = 0.90,
                                           random_state: int = RANDOM_STATE,
                                           n_jobs: int = N_JOBS):
    """
    Fit a quick RF to get importances; drop one from each highly-correlated pair.
    """
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=X.columns)

    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        high = upper.index[upper[col] > threshold].tolist()
        for row in high:
            drop = row if imp[row] < imp[col] else col
            to_drop.add(drop)

    kept = [c for c in X.columns if c not in to_drop]
    return X[kept], kept

In [65]:
def fit_fast_selector_and_transform(
    X_train: pd.DataFrame, y_train, groups_train,
    X_val: pd.DataFrame, X_test: pd.DataFrame,
    *,
    autocorr_keywords=("ATSC", "AATS"),
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
):
    """
    Feature selection pipeline for Mordred descriptors:
    1. Drop autocorrelation descriptors
    2. Variance threshold (train-only)
    3. Correlation-based pruning (train-only)
    4. Top-K by RF importance (train-only)
    5. RFECV with scaffold-aware CV (train-only for selection)
    
    All decisions made on training data only to avoid data leakage.
    """
    # 1) drop autocorr
    Xtr = drop_autocorr_cols(X_train, keywords=autocorr_keywords)
    Xva = drop_autocorr_cols(X_val, keywords=autocorr_keywords)
    Xte = drop_autocorr_cols(X_test, keywords=autocorr_keywords)

    # Ensure all splits have same columns (they should, from fit_mordred_cleaner + apply_mordred_cleaner)
    # NO data leakage: we take columns from training set only
    Xva = Xva[Xtr.columns]
    Xte = Xte[Xtr.columns]

    # 2) VT
    Xtr, Xva, Xte, kept_vt = apply_variance_threshold_train(Xtr, Xva, Xte, threshold=vt_threshold)

    # 3) correlation prune (train-only)
    Xtr, kept_corr = prune_correlated_features_by_importance(
        Xtr, y_train, threshold=corr_threshold, random_state=random_state, n_jobs=n_jobs
    )
    Xva = Xva[kept_corr]
    Xte = Xte[kept_corr]

    # 4) top-k by quick RF importance (train-only)
    rf_rank = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    rf_rank.fit(Xtr, y_train)
    imp = pd.Series(rf_rank.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
    top_k = min(top_k, len(imp))
    kept_topk = imp.head(top_k).index.tolist()

    Xtr_k = Xtr[kept_topk]
    Xva_k = Xva[kept_topk]
    Xte_k = Xte[kept_topk]

    # 5) light RFECV on the reduced set (scaffold-aware)
    base_rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    cv = GroupKFold(n_splits=rfecv_cv_splits)
    selector = RFECV(
        estimator=base_rf,
        step=rfecv_step,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=n_jobs,
        min_features_to_select=min_features,
        verbose=0
    )
    selector.fit(Xtr_k, y_train, groups=groups_train)
    kept_final = Xtr_k.columns[selector.support_].tolist()

    info = {
        "kept_vt": kept_vt,
        "kept_corr": kept_corr,
        "kept_topk": kept_topk,
        "kept_final": kept_final
    }

    return Xtr_k[kept_final], Xva_k[kept_final], Xte_k[kept_final], info, selector

In [66]:
def scaffold_cv_mordred_with_selector(
    X_all: pd.DataFrame,
    y_all,
    smiles_all,
    transformers,
    *,
    n_splits=N_SPLITS,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=4,
    rfecv_step=0.1,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
):
    """
    Scaffold-aware CV with per-fold Mordred feature selection.
    Uses fit_fast_selector_and_transform on each fold.
    """
    groups_all = np.array([murcko_scaffold(s) for s in smiles_all])
    outer = GroupKFold(n_splits=n_splits)

    maes, rmses, r2s, nfeats = [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(outer.split(X_all, y_all, groups_all), 1):
        Xtr, ytr = X_all.iloc[tr_idx], np.asarray(y_all)[tr_idx]
        Xva, yva = X_all.iloc[va_idx], np.asarray(y_all)[va_idx]
        groups_tr = groups_all[tr_idx]

        # Per-fold feature selection
        Xtr_sel, Xva_sel, _, info, _ = fit_fast_selector_and_transform(
            Xtr, ytr, groups_tr,
            Xva, Xva,  # dummy test
            vt_threshold=vt_threshold,
            corr_threshold=corr_threshold,
            top_k=top_k,
            min_features=min_features,
            rfecv_cv_splits=rfecv_cv_splits,
            rfecv_step=rfecv_step,
            random_state=random_state,
            n_jobs=n_jobs
        )

        # Train RF
        model = RandomForestRegressor(
            n_estimators=1500,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=n_jobs,
            random_state=random_state
        )
        model.fit(Xtr_sel, ytr)
        pred = model.predict(Xva_sel)

        # Evaluate
        metrics = untransform_and_evaluate(yva, pred, transformers, 
                                          prefix=f"Fold {fold}: ", verbose=True)
        maes.append(metrics["mae"])
        rmses.append(metrics["rmse"])
        r2s.append(metrics["r2"])
        nfeats.append(len(info["kept_final"]))

    # Summary
    print(f"\n{n_splits}-Fold Scaffold CV (Mordred with per-fold selection):")
    print(f"MAE  mean±std:  {np.mean(maes):.4f} ± {np.std(maes):.4f}")
    print(f"RMSE mean±std:  {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
    print(f"R²   mean±std:  {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
    print(f"n_feat mean±std: {np.mean(nfeats):.1f} ± {np.std(nfeats):.1f}")

    return {
        "maes": maes,
        "rmses": rmses,
        "r2s": r2s,
        "nfeats": nfeats,
        "mean_mae": np.mean(maes),
        "std_mae": np.std(maes),
        "mean_rmse": np.mean(rmses),
        "std_rmse": np.std(rmses),
        "mean_r2": np.mean(r2s),
        "std_r2": np.std(r2s),
    }

In [67]:
# ============================================================================
# 2D MORDRED DESCRIPTORS: EVALUATION
# ============================================================================

# Build X/y/smiles from cleaned dfs with scoped naming to avoid overwrites
X_m2d_train = mordred_df_2d_train.drop(columns=["smiles", "target"])
y_m2d_train = mordred_df_2d_train["target"].values
smiles_m2d_train = mordred_df_2d_train["smiles"].values

X_m2d_val = mordred_df_2d_val.drop(columns=["smiles", "target"])
y_m2d_val = mordred_df_2d_val["target"].values

X_m2d_test = mordred_df_2d_test.drop(columns=["smiles", "target"])
y_m2d_test = mordred_df_2d_test["target"].values

groups_m2d_train = np.array([murcko_scaffold(s) for s in smiles_m2d_train])

# Feature selection on official split (train/val/test from dataset)
Xm2d_sel, Xm2dv_sel, Xm2dt_sel, info_m2d, _ = fit_fast_selector_and_transform(
    X_m2d_train, y_m2d_train, groups_m2d_train,
    X_m2d_val, X_m2d_test,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1
)

print(f"2D Mordred: {len(info_m2d['kept_final'])} features selected for final model")

# Train final RF on official split
final_rf_m2d = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE
)
final_rf_m2d.fit(Xm2d_sel, y_m2d_train)

pred_m2d_val = final_rf_m2d.predict(Xm2dv_sel)
pred_m2d_test = final_rf_m2d.predict(Xm2dt_sel)

# Evaluate on official split
print("\n2D Mordred - Official evaluation:")
print("VAL:  ", end="")
metrics_m2d_val = untransform_and_evaluate(y_m2d_val, pred_m2d_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_m2d_test = untransform_and_evaluate(y_m2d_test, pred_m2d_test, transformers, verbose=True)

2D Mordred: 90 features selected for final model

2D Mordred - Official evaluation:
VAL:  MAE=0.5680, RMSE=0.7477, R²=0.6325
TEST: MAE=0.5946, RMSE=0.7799, R²=0.4867


In [68]:
# ============================================================================
# 2D MORDRED: 5-FOLD SCAFFOLD CV
# ============================================================================

X_all_m2d = pd.concat([X_m2d_train, X_m2d_val, X_m2d_test], axis=0)
y_all_m2d = np.concatenate([y_m2d_train, y_m2d_val, y_m2d_test])
smiles_all_m2d = np.concatenate([smiles_m2d_train, 
                                 mordred_df_2d_val["smiles"].values,
                                 mordred_df_2d_test["smiles"].values])

result_m2d_cv = scaffold_cv_mordred_with_selector(
    X_all_m2d, y_all_m2d, smiles_all_m2d, transformers
)

Fold 1: MAE=0.5491, RMSE=0.7166, R²=0.6505
Fold 2: MAE=0.5904, RMSE=0.7540, R²=0.6564
Fold 3: MAE=0.5377, RMSE=0.7039, R²=0.6447
Fold 4: MAE=0.5850, RMSE=0.7422, R²=0.6044
Fold 5: MAE=0.5477, RMSE=0.7049, R²=0.5931

5-Fold Scaffold CV (Mordred with per-fold selection):
MAE  mean±std:  0.5620 ± 0.0214
RMSE mean±std:  0.7243 ± 0.0203
R²   mean±std:  0.6298 ± 0.0259
n_feat mean±std: 118.0 ± 52.7


In [69]:
# ============================================================================
# 3D MORDRED DESCRIPTORS: EVALUATION
# ============================================================================

# Build X/y/smiles from cleaned dfs with scoped naming
X_m3d_train = mordred_df_3d_train.drop(columns=["smiles", "target"])
y_m3d_train = mordred_df_3d_train["target"].values
smiles_m3d_train = mordred_df_3d_train["smiles"].values

X_m3d_val = mordred_df_3d_val.drop(columns=["smiles", "target"])
y_m3d_val = mordred_df_3d_val["target"].values

X_m3d_test = mordred_df_3d_test.drop(columns=["smiles", "target"])
y_m3d_test = mordred_df_3d_test["target"].values

groups_m3d_train = np.array([murcko_scaffold(s) for s in smiles_m3d_train])

# Feature selection on official split
Xm3d_sel, Xm3dv_sel, Xm3dt_sel, info_m3d, _ = fit_fast_selector_and_transform(
    X_m3d_train, y_m3d_train, groups_m3d_train,
    X_m3d_val, X_m3d_test,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1
)

print(f"3D Mordred: {len(info_m3d['kept_final'])} features selected for final model")

# Train final RF on official split
final_rf_m3d = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE
)
final_rf_m3d.fit(Xm3d_sel, y_m3d_train)

pred_m3d_val = final_rf_m3d.predict(Xm3dv_sel)
pred_m3d_test = final_rf_m3d.predict(Xm3dt_sel)

# Evaluate on official split
print("\n3D Mordred - Official evaluation:")
print("VAL:  ", end="")
metrics_m3d_val = untransform_and_evaluate(y_m3d_val, pred_m3d_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_m3d_test = untransform_and_evaluate(y_m3d_test, pred_m3d_test, transformers, verbose=True)

3D Mordred: 90 features selected for final model

3D Mordred - Official evaluation:
VAL:  MAE=0.5713, RMSE=0.7420, R²=0.6339
TEST: MAE=0.5908, RMSE=0.7709, R²=0.4985


In [70]:
# ============================================================================
# 3D MORDRED: 5-FOLD SCAFFOLD CV
# ============================================================================

X_all_m3d = pd.concat([X_m3d_train, X_m3d_val, X_m3d_test], axis=0)
y_all_m3d = np.concatenate([y_m3d_train, y_m3d_val, y_m3d_test])
smiles_all_m3d = np.concatenate([smiles_m3d_train,
                                 mordred_df_3d_val["smiles"].values,
                                 mordred_df_3d_test["smiles"].values])

result_m3d_cv = scaffold_cv_mordred_with_selector(
    X_all_m3d, y_all_m3d, smiles_all_m3d, transformers
)

Fold 1: MAE=0.5576, RMSE=0.7267, R²=0.6459
Fold 2: MAE=0.6013, RMSE=0.7719, R²=0.5917
Fold 3: MAE=0.5541, RMSE=0.7038, R²=0.6275
Fold 4: MAE=0.5705, RMSE=0.7545, R²=0.6084
Fold 5: MAE=0.5441, RMSE=0.6943, R²=0.6542

5-Fold Scaffold CV (Mordred with per-fold selection):
MAE  mean±std:  0.5655 ± 0.0198
RMSE mean±std:  0.7302 ± 0.0294
R²   mean±std:  0.6256 ± 0.0231
n_feat mean±std: 144.0 ± 44.1


In [71]:
# Columns available only in the 3D Mordred representation
extra_3d_cols = sorted(set(X_m3d_train.columns) - set(X_m2d_train.columns))

# Of the features selected by the 3D selector, which ones are truly 3D-only?
selected_m3d = set(info_m3d["kept_final"])
picked_extra = sorted(selected_m3d.intersection(extra_3d_cols))
len(picked_extra), picked_extra[:20]

(6, ['FNSA3', 'FNSA5', 'FPSA4', 'PNSA1', 'RPSA', 'TASA'])

## Molecular Fingerprints

In [72]:
# ============================================================================
# FINGERPRINT COMPUTATION: MACCS, ECFP, ErG
# ============================================================================

def maccs_to_numpy(smiles):
    """Compute MACCS fingerprint as numpy array."""
    mol = Chem.MolFromSmiles(smiles)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

maccs_train = data_train['smiles'].apply(maccs_to_numpy)
maccs_val = data_val['smiles'].apply(maccs_to_numpy)
maccs_test = data_test['smiles'].apply(maccs_to_numpy)

In [73]:
# ECFP fingerprints (Morgan, radius=2, 512 bits)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)
ecfp_train = data_train['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))
ecfp_val = data_val['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))
ecfp_test = data_test['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))

In [74]:
# ErG (Gobbi pharmacophore 2D) fingerprints
factory = Gobbi_Pharm2D.factory

def erg_to_dense(smiles):
    """Convert ErG fingerprint to dense numpy array."""
    mol = Chem.MolFromSmiles(smiles)
    fp = Generate.Gen2DFingerprint(mol, factory)  # SparseBitVect
    n_bits = fp.GetNumBits()
    arr = np.zeros(n_bits, dtype=np.float32)
    for idx in fp.GetOnBits():
        arr[idx] = 1.0
    return arr

erg_train = np.vstack([erg_to_dense(s) for s in data_train["smiles"].values])
erg_val   = np.vstack([erg_to_dense(s) for s in data_val["smiles"].values])
erg_test  = np.vstack([erg_to_dense(s) for s in data_test["smiles"].values])

In [75]:
print(len(maccs_train.tolist()[0]), len(ecfp_train.tolist()[0]), len(erg_train.tolist()[0]))

167 512 39972


In [76]:
# ============================================================================
# FINGERPRINT MODELS: SGDRegressor and SVR Pipelines
# ============================================================================

def make_sgdr(alpha=1e-4, random_state=RANDOM_STATE):
    """SGDRegressor for fingerprints (MACCS, ECFP)."""
    return SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=alpha,
        max_iter=5000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=random_state
    )


def make_erg_pipeline(n_components=256, alpha=1e-4, random_state=RANDOM_STATE):
    """Pipeline for ErG: TruncatedSVD + SGDRegressor."""
    return Pipeline([
        ("svd", TruncatedSVD(n_components=n_components, random_state=random_state)),
        ("sgd", make_sgdr(alpha=alpha, random_state=random_state)),
    ])


def make_svr(C=10.0, epsilon=0.1, random_state=RANDOM_STATE):
    """SVR pipeline for embeddings (Mol2Vec, ChemBERTa)."""
    return make_pipeline(
        StandardScaler(),
        SVR(C=C, gamma="scale", epsilon=epsilon, kernel="rbf")
    )

In [77]:
# ============================================================================
# HYPERPARAMETER TUNING HELPERS
# ============================================================================

def tune_sgdr(X_train, y_train, groups_train, n_splits=5, n_jobs=N_JOBS):
    """
    Tune SGDRegressor alpha via scaffold-aware GridSearchCV.
    """
    param_grid = {'alpha': [1e-5, 1e-4, 1e-3, 1e-2]}
    
    base = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        max_iter=5000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=RANDOM_STATE
    )
    
    cv = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        base, param_grid, 
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=n_jobs,
        verbose=1
    )
    
    grid.fit(X_train, y_train, groups=groups_train)
    print(f"Best alpha: {grid.best_params_['alpha']:.2e}")
    print(f"Best CV MAE: {-grid.best_score_:.4f}")
    
    return grid.best_params_


def tune_erg_pipeline(X_train, y_train, groups_train, n_splits=5, n_jobs=N_JOBS):
    """
    Tune ErG pipeline: TruncatedSVD n_components + SGDRegressor alpha.
    """
    param_grid = {
        'svd__n_components': [128, 256, 512],
        'sgd__alpha': [1e-5, 1e-4, 1e-3, 1e-2]
    }
    
    pipeline = Pipeline([
        ("svd", TruncatedSVD(random_state=RANDOM_STATE)),
        ("sgd", SGDRegressor(
            loss="squared_error",
            penalty="l2",
            max_iter=5000,
            tol=1e-4,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=RANDOM_STATE
        ))
    ])
    
    cv = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipeline, param_grid,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=n_jobs,
        verbose=1
    )
    
    grid.fit(X_train, y_train, groups=groups_train)
    print(f"Best n_components: {grid.best_params_['svd__n_components']}")
    print(f"Best alpha: {grid.best_params_['sgd__alpha']:.2e}")
    print(f"Best CV MAE: {-grid.best_score_:.4f}")
    
    return grid.best_params_


def tune_svr(X_train, y_train, groups_train, n_splits=5, n_jobs=N_JOBS):
    """
    Tune SVR C and epsilon via scaffold-aware GridSearchCV.
    """
    param_grid = {
        'svr__C': [1, 10, 100],
        'svr__epsilon': [0.01, 0.1, 0.5]
    }
    
    pipeline = make_pipeline(
        StandardScaler(),
        SVR(gamma="scale", kernel="rbf")
    )
    
    cv = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipeline, param_grid,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=n_jobs,
        verbose=1
    )
    
    grid.fit(X_train, y_train, groups=groups_train)
    print(f"Best C: {grid.best_params_['svr__C']}")
    print(f"Best epsilon: {grid.best_params_['svr__epsilon']}")
    print(f"Best CV MAE: {-grid.best_score_:.4f}")
    
    return grid.best_params_

In [78]:
def train_and_evaluate_fp(X_train, y_train, X_val, y_val, X_test, y_test, 
                          transformers, fp_type="maccs", random_state=RANDOM_STATE):
    """
    Train and evaluate a fingerprint model on official split.
    
    Args:
        fp_type: "maccs", "ecfp", or "erg"
    """
    if fp_type == "erg":
        model = make_erg_pipeline(n_components=256, alpha=1e-4, random_state=random_state)
    else:
        model = make_sgdr(alpha=1e-4, random_state=random_state)

    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)

    print(f"{fp_type.upper()} - Official evaluation:")
    print("VAL:  ", end="")
    metrics_val = untransform_and_evaluate(y_val, pred_val, transformers, verbose=True)
    print("TEST: ", end="")
    metrics_test = untransform_and_evaluate(y_test, pred_test, transformers, verbose=True)
    
    return model, metrics_val, metrics_test

In [80]:
# ============================================================================
# FINGERPRINTS: OFFICIAL EVALUATION (train/val/test split)
# ============================================================================

# MACCS
model_maccs, _, _ = train_and_evaluate_fp(
    np.vstack(maccs_train.values), data_train["target"].values, 
    np.vstack(maccs_val.values), data_val["target"].values,
    np.vstack(maccs_test.values), data_test["target"].values,
    transformers, fp_type="maccs"
)

print()

# ECFP
model_ecfp, _, _ = train_and_evaluate_fp(
    np.vstack(ecfp_train.values), data_train["target"].values,
    np.vstack(ecfp_val.values), data_val["target"].values,
    np.vstack(ecfp_test.values), data_test["target"].values,
    transformers, fp_type="ecfp"
)

print()

# ErG
model_erg, _, _ = train_and_evaluate_fp(
    erg_train, data_train["target"].values,
    erg_val, data_val["target"].values,
    erg_test, data_test["target"].values,
    transformers, fp_type="erg"
)

MACCS - Official evaluation:
VAL:  MAE=0.8224, RMSE=1.0266, R²=0.2889
TEST: MAE=0.7963, RMSE=1.0030, R²=0.1745

ECFP - Official evaluation:
VAL:  MAE=0.8624, RMSE=1.1073, R²=0.1727
TEST: MAE=0.8292, RMSE=1.0604, R²=0.0774

ERG - Official evaluation:
VAL:  MAE=0.8115, RMSE=1.0285, R²=0.2862
TEST: MAE=0.7483, RMSE=0.9365, R²=0.2804


In [81]:
def scaffold_cv_on_array(X, y, groups, make_model, transformers, n_splits=N_SPLITS):
    """
    Simple scaffold-aware CV on precomputed feature array (numpy array or list).
    """
    # Convert to numpy array if needed
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    y = np.asarray(y)
    
    cv = GroupKFold(n_splits=n_splits)
    maes, rmses, r2s = [], [], []

    for fold, (tr, va) in enumerate(cv.split(X, y, groups), 1):
        model = make_model()
        model.fit(X[tr], y[tr])
        pred = model.predict(X[va])

        metrics = untransform_and_evaluate(y[va], pred, transformers, 
                                          prefix=f"Fold {fold}: ", verbose=True)
        maes.append(metrics["mae"])
        rmses.append(metrics["rmse"])
        r2s.append(metrics["r2"])

    print(f"\n{n_splits}-fold Scaffold CV:")
    print(f"MAE  mean±std:  {np.mean(maes):.4f} ± {np.std(maes):.4f}")
    print(f"RMSE mean±std:  {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
    print(f"R²   mean±std:  {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
    
    return {
        "maes": maes,
        "rmses": rmses,
        "r2s": r2s,
        "mean_mae": np.mean(maes),
        "std_mae": np.std(maes),
        "mean_rmse": np.mean(rmses),
        "std_rmse": np.std(rmses),
        "mean_r2": np.mean(r2s),
        "std_r2": np.std(r2s),
    }

In [82]:
# ============================================================================
# FINGERPRINTS: 5-FOLD SCAFFOLD CV
# ============================================================================

groups_fp_train = np.array([murcko_scaffold(s) for s in data_train['smiles']])

print("MACCS CV:")
result_maccs_cv = scaffold_cv_on_array(
    np.vstack(maccs_train.values), np.array(data_train["target"].values), 
    groups_fp_train, 
    lambda: make_sgdr(1e-4),
    transformers
)

print("\nECFP CV:")
result_ecfp_cv = scaffold_cv_on_array(
    np.vstack(ecfp_train.values), np.array(data_train["target"].values), 
    groups_fp_train,
    lambda: make_sgdr(1e-4),
    transformers
)

print("\nErG CV:")
result_erg_cv = scaffold_cv_on_array(
    erg_train, np.array(data_train["target"].values), 
    groups_fp_train,
    lambda: make_erg_pipeline(256, 1e-4),
    transformers
)

MACCS CV:
Fold 1: MAE=0.8037, RMSE=0.9948, R²=0.3532
Fold 2: MAE=0.7931, RMSE=1.0138, R²=0.3932
Fold 3: MAE=0.7876, RMSE=0.9869, R²=0.2103
Fold 4: MAE=0.7173, RMSE=0.8980, R²=0.3304
Fold 5: MAE=0.8156, RMSE=1.0244, R²=0.3117

5-fold Scaffold CV:
MAE  mean±std:  0.7834 ± 0.0344
RMSE mean±std:  0.9836 ± 0.0448
R²   mean±std:  0.3198 ± 0.0611

ECFP CV:
Fold 1: MAE=0.8309, RMSE=1.0505, R²=0.2787
Fold 2: MAE=0.8731, RMSE=1.1097, R²=0.2729
Fold 3: MAE=0.7897, RMSE=1.0002, R²=0.1888
Fold 4: MAE=0.7609, RMSE=0.9679, R²=0.2220
Fold 5: MAE=0.8472, RMSE=1.0537, R²=0.2718

5-fold Scaffold CV:
MAE  mean±std:  0.8204 ± 0.0402
RMSE mean±std:  1.0364 ± 0.0487
R²   mean±std:  0.2469 ± 0.0355

ErG CV:
Fold 1: MAE=0.7893, RMSE=1.0134, R²=0.3288
Fold 2: MAE=0.8141, RMSE=1.0230, R²=0.3821
Fold 3: MAE=0.7426, RMSE=0.9179, R²=0.3168
Fold 4: MAE=0.6794, RMSE=0.8637, R²=0.3806
Fold 5: MAE=0.7571, RMSE=0.9777, R²=0.3730

5-fold Scaffold CV:
MAE  mean±std:  0.7565 ± 0.0459
RMSE mean±std:  0.9591 ± 0.0603
R²   me

## Mol2Vec

In [83]:
# ============================================================================
# MOL2VEC EMBEDDINGS & EVALUATION
# ============================================================================

def sentences2vec(sentences, model, unseen='UNK'):
    """
    Convert a list of sentences (molecular substructure tokens) into vector representations
    using a gensim Word2Vec model.
    
    Args:
        sentences: list of lists of substructure tokens
        model: gensim Word2Vec model
        unseen: token to use for unseen words
    
    Returns:
        list of numpy arrays (one vector per molecule)
    """
    vectors = []
    for sentence in tqdm(sentences, desc="Embedding molecules"):
        vecs = []
        for word in sentence:
            if word in model.wv:
                vecs.append(model.wv[word])
            elif unseen in model.wv:
                vecs.append(model.wv[unseen])
        if vecs:
            vectors.append(np.mean(vecs, axis=0))
        else:
            vectors.append(np.zeros(model.vector_size))
    return vectors

In [84]:
# Load pre-trained Mol2Vec model
model_m2v = gensim.models.Word2Vec.load("models/model_300dim.pkl")

# Compute sentences (molecular substructure tokens)
sentences_train = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_train['smiles'].tolist()]
sentences_val = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_val['smiles'].tolist()]
sentences_test = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_test['smiles'].tolist()]

# Convert to vectors
vectors_m2v_train = np.array(sentences2vec(sentences_train, model_m2v))
vectors_m2v_val = np.array(sentences2vec(sentences_val, model_m2v))
vectors_m2v_test = np.array(sentences2vec(sentences_test, model_m2v))

Embedding molecules: 100%|██████████| 420/420 [00:00<00:00, 4441.34it/s]


In [85]:
# ============================================================================
# MOL2VEC: OFFICIAL EVALUATION (train/val/test split)
# ============================================================================

# Build arrays with scoped naming
X_m2v_train = np.vstack(vectors_m2v_train)
y_m2v_train = data_train["target"].values
X_m2v_val = np.vstack(vectors_m2v_val)
y_m2v_val = data_val["target"].values
X_m2v_test = np.vstack(vectors_m2v_test)
y_m2v_test = data_test["target"].values

# Train SVR
svr_m2v = make_svr(C=10.0, epsilon=0.1)
svr_m2v.fit(X_m2v_train, y_m2v_train)

pred_m2v_val = svr_m2v.predict(X_m2v_val)
pred_m2v_test = svr_m2v.predict(X_m2v_test)

# Evaluate
print("Mol2Vec - Official evaluation:")
print("VAL:  ", end="")
metrics_m2v_val = untransform_and_evaluate(y_m2v_val, pred_m2v_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_m2v_test = untransform_and_evaluate(y_m2v_test, pred_m2v_test, transformers, verbose=True)

Mol2Vec - Official evaluation:
VAL:  MAE=0.5742, RMSE=0.7407, R²=0.6298
TEST: MAE=0.6067, RMSE=0.7766, R²=0.5051


In [86]:
# ============================================================================
# MOL2VEC: 5-FOLD SCAFFOLD CV
# ============================================================================

X_all_m2v = np.vstack([vectors_m2v_train, vectors_m2v_val, vectors_m2v_test])
y_all_m2v = np.concatenate([data_train["target"].values,
                            data_val["target"].values,
                            data_test["target"].values])
smiles_all_m2v = np.concatenate([data_train["smiles"].values, 
                                 data_val["smiles"].values,
                                 data_test["smiles"].values])

result_m2v_cv = scaffold_cv(
    X_all_m2v, y_all_m2v, smiles_all_m2v,
    lambda: make_svr(C=10.0, epsilon=0.1),
    transformers,
    n_splits=N_SPLITS
)

Fold 1: MAE=0.5627, RMSE=0.7516, R²=0.6458
Fold 2: MAE=0.5703, RMSE=0.7483, R²=0.6248
Fold 3: MAE=0.5592, RMSE=0.7289, R²=0.5723
Fold 4: MAE=0.5389, RMSE=0.7237, R²=0.5971
Fold 5: MAE=0.5800, RMSE=0.7671, R²=0.6174

5-Fold Scaffold CV Summary:
MAE  mean±std:  0.5622 ± 0.0137
RMSE mean±std:  0.7439 ± 0.0158
R²   mean±std:  0.6115 ± 0.0250


# ChemBERTa

In [87]:
# ============================================================================
# CHEMBERTA EMBEDDINGS FUNCTION
# ============================================================================

def chemberta_embeddings(smiles_list, model_name="seyonec/ChemBERTa-zinc-base-v1",
                         batch_size=32, max_length=256, device=None):
    """
    Compute ChemBERTa embeddings for a list of SMILES strings.
    Uses mean pooling over tokens.
    """
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Ensure list[str]
    if not isinstance(smiles_list, list):
        smiles_list = smiles_list.tolist()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_vecs = []
    with torch.no_grad():
        for i in range(0, len(smiles_list), batch_size):
            batch = smiles_list[i:i+batch_size]

            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            out = model(**enc)
            pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
            all_vecs.append(pooled.cpu().numpy().astype(np.float32))

    return np.vstack(all_vecs)

In [88]:
# ============================================================================
# CHEMBERTA PRETRAINED: EMBEDDINGS & EVALUATION
# ============================================================================

# Get SMILES from official split (use the global data_train/val/test)
smiles_cb_train = data_train["smiles"].values
smiles_cb_val   = data_val["smiles"].values
smiles_cb_test  = data_test["smiles"].values

y_cb_train = data_train["target"].values
y_cb_val   = data_val["target"].values
y_cb_test  = data_test["target"].values

# Compute embeddings using pretrained ChemBERTa
print("Computing ChemBERTa (pretrained) embeddings...")
X_cb_train = chemberta_embeddings(smiles_cb_train, model_name="seyonec/ChemBERTa-zinc-base-v1")
X_cb_val   = chemberta_embeddings(smiles_cb_val,   model_name="seyonec/ChemBERTa-zinc-base-v1")
X_cb_test  = chemberta_embeddings(smiles_cb_test,  model_name="seyonec/ChemBERTa-zinc-base-v1")

# Train SVR
svr_cb = make_svr(C=10.0, epsilon=0.1)
svr_cb.fit(X_cb_train, y_cb_train)

pred_cb_val = svr_cb.predict(X_cb_val)
pred_cb_test = svr_cb.predict(X_cb_test)

# Evaluate
print("ChemBERTa (pretrained) - Official evaluation:")
print("VAL:  ", end="")
metrics_cb_val = untransform_and_evaluate(y_cb_val, pred_cb_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_cb_test = untransform_and_evaluate(y_cb_test, pred_cb_test, transformers, verbose=True)

Computing ChemBERTa (pretrained) embeddings...


c:\Users\afs\anaconda3\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


ChemBERTa (pretrained) - Official evaluation:
VAL:  MAE=0.7432, RMSE=0.9774, R²=0.3555
TEST: MAE=0.6675, RMSE=0.8550, R²=0.4002


In [89]:
# ============================================================================
# CHEMBERTA PRETRAINED: 5-FOLD SCAFFOLD CV
# ============================================================================

X_all_cb = np.vstack([X_cb_train, X_cb_val, X_cb_test])
y_all_cb = np.concatenate([y_cb_train, y_cb_val, y_cb_test])
smiles_all_cb = np.concatenate([smiles_cb_train, smiles_cb_val, smiles_cb_test])

result_cb_cv = scaffold_cv(
    X_all_cb, y_all_cb, smiles_all_cb,
    lambda: make_svr(C=10.0, epsilon=0.1),
    transformers,
    n_splits=N_SPLITS
)

Fold 1: MAE=0.7310, RMSE=0.9421, R²=0.4436
Fold 2: MAE=0.7707, RMSE=0.9756, R²=0.3623
Fold 3: MAE=0.6848, RMSE=0.8718, R²=0.3882
Fold 4: MAE=0.6712, RMSE=0.8623, R²=0.4280
Fold 5: MAE=0.6937, RMSE=0.9052, R²=0.4671

5-Fold Scaffold CV Summary:
MAE  mean±std:  0.7103 ± 0.0361
RMSE mean±std:  0.9114 ± 0.0426
R²   mean±std:  0.4179 ± 0.0378


### ChemBERTa training

In [37]:
# !pip install -U "accelerate>=0.26.0"

In [90]:
# ============================================================================
# CHEMBERTA: MLM FINE-TUNING ON TRAINING DATA
# ============================================================================

import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# 1) Load SMILES corpus from training split ONLY (to avoid leakage)
smiles_corpus_finetuned = data_train["smiles"].astype(str).tolist()

# 2) Tokenizer + MLM model
model_name_base = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer_cb = AutoTokenizer.from_pretrained(model_name_base)
model_cb = AutoModelForMaskedLM.from_pretrained(model_name_base)

# Optional: memory savers
model_cb.gradient_checkpointing_enable()

# 3) Dataset -> tokenized
ds = Dataset.from_dict({"text": smiles_corpus_finetuned})

def tokenize(batch):
    return tokenizer_cb(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])

# 4) MLM collator
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_cb,
    mlm=True,
    mlm_probability=0.15
)

# 5) Training args (GPU-friendly)
args = TrainingArguments(
    output_dir="./chemberta_mlm_ckpts",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model_cb,
    args=args,
    train_dataset=ds_tok,
    data_collator=collator,
)

print("Starting MLM fine-tuning on training SMILES corpus...")
trainer.train()

# 6) Save final model + tokenizer
final_dir = "./chemberta_mlm_final"
trainer.save_model(final_dir)
tokenizer_cb.save_pretrained(final_dir)
print(f"Fine-tuned ChemBERTa saved to: {final_dir}")

Some weights of the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

Starting MLM fine-tuning on training SMILES corpus...


Step,Training Loss
50,0.845500
100,0.600900
150,0.544800
200,0.525800
250,0.485100
300,0.504800


Fine-tuned ChemBERTa saved to: ./chemberta_mlm_final


In [91]:
# ============================================================================
# CHEMBERTA FINE-TUNED: EMBEDDINGS
# ============================================================================

print("Computing ChemBERTa (fine-tuned) embeddings...")
X_cbft_train = chemberta_embeddings(smiles_cb_train, model_name="./chemberta_mlm_final")
X_cbft_val = chemberta_embeddings(smiles_cb_val,   model_name="./chemberta_mlm_final")
X_cbft_test = chemberta_embeddings(smiles_cb_test,  model_name="./chemberta_mlm_final")

Some weights of RobertaModel were not initialized from the model checkpoint at ./chemberta_mlm_final and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Computing ChemBERTa (fine-tuned) embeddings...


Some weights of RobertaModel were not initialized from the model checkpoint at ./chemberta_mlm_final and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ./chemberta_mlm_final and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [92]:
# ============================================================================
# CHEMBERTA FINE-TUNED: OFFICIAL EVALUATION (train/val/test split)
# ============================================================================

# Train SVR on fine-tuned embeddings
svr_cbft = make_svr(C=10.0, epsilon=0.1)
svr_cbft.fit(X_cbft_train, y_cb_train)

pred_cbft_val = svr_cbft.predict(X_cbft_val)
pred_cbft_test = svr_cbft.predict(X_cbft_test)

# Evaluate
print("ChemBERTa (fine-tuned) - Official evaluation:")
print("VAL:  ", end="")
metrics_cbft_val = untransform_and_evaluate(y_cb_val, pred_cbft_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_cbft_test = untransform_and_evaluate(y_cb_test, pred_cbft_test, transformers, verbose=True)

ChemBERTa (fine-tuned) - Official evaluation:
VAL:  MAE=0.7375, RMSE=0.9608, R²=0.3772
TEST: MAE=0.6838, RMSE=0.8852, R²=0.3571


### Note on ChemBERTa Fine-Tuned CV

**Why no 5-fold scaffold CV for fine-tuned ChemBERTa:**

The fine-tuning was performed using the **training split only** (via MLM pretraining on `data_train["smiles"]`). If we were to run 5-fold scaffold cross-validation on the combined dataset (train + val + test), some molecules in the fine-tuned model's training corpus would appear in the CV validation folds. This constitutes data leakage and would artificially inflate CV performance metrics.

**Reported metrics:** Only official split metrics (val and test on the original DeepChem split) are reported for fine-tuned ChemBERTa, which correctly avoid this leakage.

For a fair comparison with other models, the **pretrained ChemBERTa** (not fine-tuned) is used for the full 5-fold scaffold CV.

# Feature Fusion: Combining All Representations

This is the central experiment of the thesis: does combining molecular descriptors, fingerprints, Mol2Vec, and ChemBERTa embeddings into one fused representation outperform any single representation alone? The sections above evaluate each representation in isolation; this section concatenates them and re-runs the same leakage-safe pipeline.

**Fusion strategy:** early fusion (concatenation), followed by the same per-fold feature selection (`fit_fast_selector_and_transform`) and scaffold CV (`scaffold_cv_mordred_with_selector`) already used for the Mordred descriptors above, so the fused model is evaluated under identical conditions.

**Representations combined:** 2D Mordred descriptors, 3D Mordred descriptors, MACCS keys, ECFP/Morgan fingerprints, ErG pharmacophore fingerprints (SVD-reduced), Mol2Vec, and **pretrained** ChemBERTa embeddings.

**ChemBERTa fine-tuned (CB-FT) is excluded** from this CV-based fused model, for the same reason it's excluded from the full-dataset scaffold CV above: its embeddings come from a model MLM-fine-tuned on the training split's own SMILES, so folding it into a CV that reassigns some of that data to validation folds would leak information. A supplementary fused+CB-FT variant, evaluated only on the untouched official test split, is included further below.

**Row alignment:** fingerprints, Mol2Vec, and ChemBERTa are computed for every row in `data_train`/`data_val`/`data_test`, so they're positionally aligned with them. The Mordred 2D/3D cleaning steps (`fit_mordred_cleaner`/`apply_mordred_cleaner`) can each drop a *different* subset of molecules (failed 3D embedding, excessive missing descriptors), so all representations are joined on `smiles` below rather than concatenated positionally — this is asserted/validated (`validate="one_to_one"`) rather than assumed.

In [ ]:
# ============================================================================
# FUSION: BUILD PER-SPLIT FEATURE BLOCKS
# ============================================================================

def _fp_block(split_df, maccs_s, ecfp_s, erg_arr, m2v_arr, cb_arr):
    """
    Assemble the non-Mordred representations (positionally aligned with
    split_df, since none of them drop rows) into one DataFrame keyed by
    'smiles', so they can be joined against the Mordred descriptor frames
    (which may have dropped rows during cleaning).
    """
    n = len(split_df)
    assert len(maccs_s) == n and len(ecfp_s) == n and len(erg_arr) == n \
        and len(m2v_arr) == n and len(cb_arr) == n, "Row count mismatch before fusion"

    maccs_arr = np.vstack(maccs_s.values)
    ecfp_arr = np.vstack(ecfp_s.values)

    block = pd.concat([
        pd.DataFrame(maccs_arr, columns=[f"maccs_{i}" for i in range(maccs_arr.shape[1])]),
        pd.DataFrame(ecfp_arr, columns=[f"ecfp_{i}" for i in range(ecfp_arr.shape[1])]),
        pd.DataFrame(erg_arr, columns=[f"erg_{i}" for i in range(erg_arr.shape[1])]),
        pd.DataFrame(m2v_arr, columns=[f"m2v_{i}" for i in range(m2v_arr.shape[1])]),
        pd.DataFrame(cb_arr, columns=[f"cb_{i}" for i in range(cb_arr.shape[1])]),
    ], axis=1)
    block.insert(0, "smiles", split_df["smiles"].reset_index(drop=True).values)
    return block


def _prefix_descriptor_cols(df, prefix):
    """Prefix only the descriptor columns (not smiles/target) to avoid 2D/3D name collisions."""
    feat_cols = [c for c in df.columns if c not in ("smiles", "target")]
    return df.rename(columns={c: f"{prefix}{c}" for c in feat_cols})


def build_fused_features(m2d_df, m3d_df, fp_block):
    """
    Inner-join Mordred 2D + Mordred 3D + the fingerprint/Mol2Vec/ChemBERTa
    block on 'smiles'. Only molecules present in ALL representations survive,
    guarding against the Mordred cleaning steps dropping different molecules
    in 2D vs 3D. `validate="one_to_one"` raises if smiles aren't unique per
    split or if the join isn't clean.
    """
    m2d_p = _prefix_descriptor_cols(m2d_df, "m2d_")
    m3d_p = _prefix_descriptor_cols(m3d_df.drop(columns=["target"]), "m3d_")

    merged = m2d_p.merge(m3d_p, on="smiles", how="inner", validate="one_to_one")
    merged = merged.merge(fp_block, on="smiles", how="inner", validate="one_to_one")
    return merged


fp_block_train = _fp_block(data_train, maccs_train, ecfp_train, erg_train, vectors_m2v_train, X_cb_train)
fp_block_val   = _fp_block(data_val,   maccs_val,   ecfp_val,   erg_val,   vectors_m2v_val,   X_cb_val)
fp_block_test  = _fp_block(data_test,  maccs_test,  ecfp_test,  erg_test,  vectors_m2v_test,  X_cb_test)

fused_train_df = build_fused_features(mordred_df_2d_train, mordred_df_3d_train, fp_block_train)
fused_val_df   = build_fused_features(mordred_df_2d_val,   mordred_df_3d_val,   fp_block_val)
fused_test_df  = build_fused_features(mordred_df_2d_test,  mordred_df_3d_test,  fp_block_test)

print(f"Fused train/val/test molecule counts after inner join: "
      f"{len(fused_train_df)}/{len(fused_val_df)}/{len(fused_test_df)}")
print(f"(compare to pre-fusion split sizes: {len(data_train)}/{len(data_val)}/{len(data_test)})")
print(f"Fused feature dimensionality (pre-selection): {fused_train_df.shape[1] - 2}")  # minus smiles, target


In [ ]:
X_fused_train = fused_train_df.drop(columns=["smiles", "target"])
y_fused_train = fused_train_df["target"].values
smiles_fused_train = fused_train_df["smiles"].values

X_fused_val = fused_val_df.drop(columns=["smiles", "target"])
y_fused_val = fused_val_df["target"].values

X_fused_test = fused_test_df.drop(columns=["smiles", "target"])
y_fused_test = fused_test_df["target"].values

groups_fused_train = np.array([murcko_scaffold(s) for s in smiles_fused_train])


In [ ]:
# ============================================================================
# FUSED REPRESENTATION: EVALUATION (official split)
# ============================================================================

Xf_sel, Xfv_sel, Xft_sel, info_fused, _ = fit_fast_selector_and_transform(
    X_fused_train, y_fused_train, groups_fused_train,
    X_fused_val, X_fused_test,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1
)

print(f"Fused representation: {len(info_fused['kept_final'])} features selected "
      f"(from {X_fused_train.shape[1]} concatenated features)")

final_rf_fused = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE
)
final_rf_fused.fit(Xf_sel, y_fused_train)

pred_fused_val = final_rf_fused.predict(Xfv_sel)
pred_fused_test = final_rf_fused.predict(Xft_sel)

print("\nFused representation - Official evaluation:")
print("VAL:  ", end="")
metrics_fused_val = untransform_and_evaluate(y_fused_val, pred_fused_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_fused_test = untransform_and_evaluate(y_fused_test, pred_fused_test, transformers, verbose=True)


In [ ]:
# ============================================================================
# FUSED REPRESENTATION: 5-FOLD SCAFFOLD CV
# ============================================================================

X_all_fused = pd.concat([X_fused_train, X_fused_val, X_fused_test], axis=0)
y_all_fused = np.concatenate([y_fused_train, y_fused_val, y_fused_test])
smiles_all_fused = np.concatenate([
    smiles_fused_train, fused_val_df["smiles"].values, fused_test_df["smiles"].values
])

result_fused_cv = scaffold_cv_mordred_with_selector(
    X_all_fused, y_all_fused, smiles_all_fused, transformers
)


In [ ]:
# ============================================================================
# SUMMARY: DOES FUSION HELP? (5-fold scaffold CV, full dataset)
# ============================================================================
# MACCS/ECFP/ErG are intentionally left out of this table: their scaffold CV
# in the FINGERPRINTS section above only used the training split, not the
# full train+val+test set like the others here, so the numbers aren't
# directly comparable as-is. Worth rerunning them on the full set if you
# want a complete head-to-head.

comparison = {
    "2D Descriptors": result_m2d_cv,
    "3D Descriptors": result_m3d_cv,
    "Mol2Vec": result_m2v_cv,
    "ChemBERTa (pretrained)": result_cb_cv,
    "Fused (all combined)": result_fused_cv,
}

print(f"{'Representation':<24}{'MAE':>10}{'RMSE':>10}{'R2':>10}")
for name, res in comparison.items():
    print(f"{name:<24}{res['mean_mae']:>10.4f}{res['mean_rmse']:>10.4f}{res['mean_r2']:>10.4f}")


## Supplementary: Fusion + Fine-Tuned ChemBERTa (official split only)

This variant matches the thesis more closely, since fine-tuned ChemBERTa was one of the NLP embeddings originally used. It can only be evaluated on the untouched official test split, not via full-dataset scaffold CV, for the leakage reason explained above and in the "Note on ChemBERTa Fine-Tuned CV" cell at the end of this notebook.

In [ ]:
# ============================================================================
# FUSED REPRESENTATION + FINE-TUNED CHEMBERTA (official split only)
# ============================================================================

def _cb_block(smiles_arr, cb_arr, prefix):
    cols = {f"{prefix}_{i}": cb_arr[:, i] for i in range(cb_arr.shape[1])}
    return pd.DataFrame({"smiles": smiles_arr, **cols})

cbft_train_block = _cb_block(smiles_cb_train, X_cbft_train, "cbft")
cbft_val_block   = _cb_block(smiles_cb_val,   X_cbft_val,   "cbft")
cbft_test_block  = _cb_block(smiles_cb_test,  X_cbft_test,  "cbft")

fused_cbft_train_df = fused_train_df.merge(cbft_train_block, on="smiles", how="inner", validate="one_to_one")
fused_cbft_val_df   = fused_val_df.merge(cbft_val_block,   on="smiles", how="inner", validate="one_to_one")
fused_cbft_test_df  = fused_test_df.merge(cbft_test_block,  on="smiles", how="inner", validate="one_to_one")

X_fcbft_train = fused_cbft_train_df.drop(columns=["smiles", "target"])
y_fcbft_train = fused_cbft_train_df["target"].values
groups_fcbft_train = np.array([murcko_scaffold(s) for s in fused_cbft_train_df["smiles"]])

X_fcbft_val = fused_cbft_val_df.drop(columns=["smiles", "target"])
y_fcbft_val = fused_cbft_val_df["target"].values

X_fcbft_test = fused_cbft_test_df.drop(columns=["smiles", "target"])
y_fcbft_test = fused_cbft_test_df["target"].values

print(f"Fused+CB-FT train/val/test counts after join: "
      f"{len(fused_cbft_train_df)}/{len(fused_cbft_val_df)}/{len(fused_cbft_test_df)}")

Xfcbft_sel, Xfcbftv_sel, Xfcbftt_sel, info_fcbft, _ = fit_fast_selector_and_transform(
    X_fcbft_train, y_fcbft_train, groups_fcbft_train,
    X_fcbft_val, X_fcbft_test,
    vt_threshold=1e-8, corr_threshold=0.90, top_k=300, min_features=50,
    rfecv_cv_splits=5, rfecv_step=0.1
)
print(f"Fused + CB-FT: {len(info_fcbft['kept_final'])} features selected")

final_rf_fcbft = RandomForestRegressor(
    n_estimators=1500, max_depth=None, min_samples_leaf=2,
    n_jobs=N_JOBS, random_state=RANDOM_STATE
)
final_rf_fcbft.fit(Xfcbft_sel, y_fcbft_train)

pred_fcbft_val = final_rf_fcbft.predict(Xfcbftv_sel)
pred_fcbft_test = final_rf_fcbft.predict(Xfcbftt_sel)

print("\nFused + CB-FT - Official evaluation (TEST is the number that's safe to report; "
      "VAL used training-corpus-adjacent SMILES so treat it as optimistic):")
print("VAL:  ", end="")
metrics_fcbft_val = untransform_and_evaluate(y_fcbft_val, pred_fcbft_val, transformers, verbose=True)
print("TEST: ", end="")
metrics_fcbft_test = untransform_and_evaluate(y_fcbft_test, pred_fcbft_test, transformers, verbose=True)
